# Load AlgoWiki pages

In [1]:
import requests

In [2]:
from functools import lru_cache

@lru_cache(maxsize=1000)
def load_page(url):
    return requests.get(url)

In [3]:
algowiki_base = "https://algowiki-project.org"
algowiki_valid_pages_index_url = "/ru/Категория:Уровень_алгоритма"

In [4]:
algowiki_valid_pages_index = load_page(algowiki_base + algowiki_valid_pages_index_url)

In [5]:
from bs4 import BeautifulSoup

In [6]:
html = BeautifulSoup(algowiki_valid_pages_index.text)

In [7]:
index_section = html.find_all("div", attrs={'id':"mw-pages"})[0]
index_items = [(li.a['title'], li.a['href']) for li in index_section.find_all('li')]

In [8]:
index_items[0]

('Участник:A.Freeman/Алгоритм Ланцоша для точной арифметики (без переортогонализации)',
 '/ru/%D0%A3%D1%87%D0%B0%D1%81%D1%82%D0%BD%D0%B8%D0%BA:A.Freeman/%D0%90%D0%BB%D0%B3%D0%BE%D1%80%D0%B8%D1%82%D0%BC_%D0%9B%D0%B0%D0%BD%D1%86%D0%BE%D1%88%D0%B0_%D0%B4%D0%BB%D1%8F_%D1%82%D0%BE%D1%87%D0%BD%D0%BE%D0%B9_%D0%B0%D1%80%D0%B8%D1%84%D0%BC%D0%B5%D1%82%D0%B8%D0%BA%D0%B8_(%D0%B1%D0%B5%D0%B7_%D0%BF%D0%B5%D1%80%D0%B5%D0%BE%D1%80%D1%82%D0%BE%D0%B3%D0%BE%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D0%B8)')

In [9]:
def encode_str(text):
    return sha256(text.encode('utf-8')).hexdigest()

In [10]:
from pathlib import Path
from hashlib import sha256

def dump_html(html,path):
    with open(path, encoding='utf-8', mode='w') as f:
        f.write(html)

def load_html(path):
    path = Path(path)
    if path.exists():
        with open(path) as f:
            return f.read()
    else:
        return None

In [11]:
from tqdm.auto import tqdm
from time import sleep

OUT_DIR = "./data/html"
OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(exist_ok=True,parents=True)

for title, url in tqdm(index_items):
    name = encode_str(title + url)
    path = OUT_DIR / f"{name}.html"
    if load_html(path) is None:
        page_raw_html = load_page(algowiki_base + url).text
        dump_html(page_raw_html, path)
        sleep(0.2)

/home/orbis/anaconda3/envs/autoprompt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [00:00<00:00, 11212.92it/s]


# Parse Page

In [12]:
def takefirst(arr):
    return next(
        chain(arr,[None])
    )

In [13]:
import re

HTML_MATH_PATTERN = r"\[math\]\\displaystyle{[\n\s]*(.+?)[\n\s]*}\[/math\]"
HTML_MATH_PATTERN = re.compile(HTML_MATH_PATTERN, flags=re.M | re.U | re.DOTALL)

def html_to_latex(latex_text):
    res = HTML_MATH_PATTERN.sub(r"$$\1$$",latex_text)
    res = res.replace(" $$"," $").replace("$$ ","$ ")
    return res

In [14]:
from dataclasses import dataclass

@dataclass
class AlgoTable:
    complexity:str = "-1"
    inp_size:str = "-1"
    out_size:str = "-1"
    form_height:str = "-1"
    form_width:str = "-1"

algotable_name_map  = {
    'Последовательная сложность': "complexity",
    'Объём входных данных': "inp_size",
    'Объём выходных данных': "out_size",
    'Высота ярусно-параллельной формы': "form_height",
    'Ширина ярусно-параллельной формы': "form_width",
}

def extract_algo_properties(algo_properties_table):
    prop_dict = {}

    tmp = algo_properties_table.find_all('td', attrs={"style":"background-color:#eef; padding: 0.2em; font-weight: bold;"})
    for prop in tmp:
        prop_name = prop.text.strip()
        for s in prop.next_siblings:
            if s.text.strip() and s.span:
                prop_value = html_to_latex(s.span.text)
                prop_dict[prop_name] = prop_value
                break
    arg_dict = {algotable_name_map[k]:v for k,v in prop_dict.items() if k in algotable_name_map}
    res = AlgoTable(**arg_dict)
    return res

In [15]:
class SectionNode:
    def __init__(self, level=0, name="", text=""):
        self.level = level
        self.name=name
        self.text=text
        self.parent=None
        self.children = []

    def add_child(self, node):
        self.children.append(node)
        node._set_parent(self)

    def insert_child(self, node):
        if node.level > self.level:
            if node.level - 1 == self.level or not self.children:
                self.add_child(node)
            else:
                self.children[-1].insert_child(node)

    def _set_parent(self,node):
        self.parent = node

    def __repr__(self):
        return self.compile_text()
    
    def compile_text(self, max_depth=-1, new_level=-1):
        max_depth-=1
        level = self.level if new_level < 0 else new_level
        title = "#"*level + f" {self.name}"
        return "\n".join(
            [title, self.text, *(
                ch.compile_text(max_depth, new_level=new_level+1) for ch in self.children if max_depth != 0
            )])

    def trim_empty_leaves(self):
        retain = []
        for ch in self.children:
            if (ch.children or ch.text):
                retain.append(ch)
                ch.trim_empty_leaves()
            # else:
            #     print(ch)
        self.children=retain

In [16]:
import re

code_section_pattern = "(```.{8,}?```)"
code_section_pattern = re.compile(code_section_pattern, flags=re.M | re.DOTALL | re.U)

def find_code_sections(text):
    return list(code_section_pattern.findall(text))


def mask_substrs(text, substrs):
    mask_dict = {}
    for s, sub in enumerate(substrs):
        mask = f"[SUB{s}]"
        mask = f"{mask}{'?'*(len(sub)-2*len(mask))}{mask}"
        assert len(mask) == len(sub)
        mask_dict[mask]=sub
        text = text.replace(sub, mask)
    return text, mask_dict

def unmask_text(text, mask_dict):
    for mask, sub in mask_dict.items():
        text = text.replace(mask, sub)
    return text

In [17]:
from html_to_markdown import convert

import re

section_pattern = "(#+ )(.+?)\n+"#\n"
section_pattern = re.compile(section_pattern, flags=re.M | re.DOTALL | re.U)

def extract_sections(markdown):
    code_strs = find_code_sections(markdown)
    markdown, mask_dict = mask_substrs(markdown, code_strs)
    tmp=list(section_pattern.finditer(markdown))
    
    levels, names = list(zip(*((match.group(1), match.group(2)) for match in tmp)))
    borders = list(chain(*((match.start(), match.end()) for match in tmp)))[1:] + [None]
    borders = [slice(l,r) for l,r in zip(borders[::2], borders[1::2])]
    markdown = unmask_text(markdown, mask_dict)
    
    sections = [(len(l.strip()), n, markdown[b]) for l,n,b in zip(levels, names, borders)]
    return sections

In [18]:
def build_section_tree(sections, trim_empty=True):
    tree = SectionNode(name="ROOT")
    
    node_list = [tree]
    for level, name, text in sections:
        node = SectionNode(level, name, text)
        # print(name, level)
        tree.insert_child(node)
        node_list.append(node)
    tree.trim_empty_leaves()
    return node_list

In [19]:
empty_node = SectionNode()

def match_node(query, node_list):
    return next(
        chain(filter(
            lambda x: re.match(query.lower(), x.name.lower()),
            node_list
        ),
              [empty_node])
    )

In [20]:
def extract_title(html):
    return html.find_all("h1", attrs={"id":"firstHeading"})[0].text

In [21]:
from html_to_markdown import convert

def build_algowiki_example(page):
    html = BeautifulSoup(page)
    title = extract_title(html)
    algo_properties_table = takefirst((cand for cand in html.find_all("table") if 'Объём выходных данных' in cand.text))

    if algo_properties_table is None:
        return title, None, None
    target = extract_algo_properties(algo_properties_table)
    
    markdown = html_to_latex(convert(page).content)
    # print(markdown)
    sections = extract_sections(markdown)
    section_tree = build_section_tree(sections)
    
    impl_name = "программная реализация"
    impl_text = match_node(impl_name, section_tree)
    if not impl_text:
        print("!!!", title)
        impl_text=None
    else:
        impl_text = impl_text.compile_text(new_level=1)
        if '```' not in impl_text:
            print("!", title)
            # print(impl_text)
            impl_text=None
    
    math_desc_names = ["Общее описание алгоритма", "Математическое описание алгоритма", "Вычислительное ядро алгоритма"]
    math_desc_text = "\n".join(
        [match_node(name, section_tree).compile_text(1, new_level=1) for name in math_desc_names]
    )

    input_data = {
        "math" : math_desc_text,
        "code": impl_text
    }
    return title, target, input_data

# Build dataset

In [22]:
from dataclasses import dataclass

@dataclass
class AlgoWikiRow:
    title:str
    math_desc:str
    code_desc:str
    table : AlgoTable = None

In [23]:
from itertools import chain

data_dir = Path("./data/html/")

examples = []
for fp in data_dir.glob("*.html"):
    page=load_html(fp)
    title, target, input_data =  build_algowiki_example(page)
    if target is None:
        print("???", title)
        continue
    row = AlgoWikiRow(
        title, 
        input_data['math'],
        input_data['code'],
        target
    )
    examples.append(row)

??? LU-разложение методом Гаусса без перестановок
??? Компактная схема метода Гаусса для трёхдиагональной матрицы и её модификации
??? Методы решения СЛАУ с трёхдиагональными матрицами
??? Метод Якоби (вращений) для симметричных матриц с циклическим исключением и барьерами
! Участник:Alexander34396/Обобщенный метод минимальных невязок
! Участник:Maria Zaitseva/PAM (Partitioning Around Medoids)
??? Участник:Мязина Екатерина/Алгоритм концептуальной кластеризации COBWEB
! Участник:VolkovNikita94/Алгоритм Ланцоша для точной арифметики (без переортогонализации)
! Участник:Noite/EM-алгоритм кластеризации
! Алгоритм Пурдома
! Алговики:Используемые шаблоны
! Алгоритм Холецкого
! Участник:Igor.orpanen/Плотностный алгоритм кластеризации (DBSCAN)
??? LU-разложение методом Гаусса с выбором ведущего элемента по главной диагонали
??? LU-разложение методом Гаусса с выбором ведущего элемента по столбцу
! Участник:Антон Тодуа/Partitioning Around Medoids (PAM)
! Участник:Kaholicl/Адаптивный крестовый ме

In [24]:
print(examples[1].code_desc)

# Программная реализация алгоритма

## Особенности реализации последовательного алгоритма
На языке C функцию однокубитного преобразования можно записать следующим образом:

```
void OneQubitEvolution(complexd *in, complexd *out, complexd U[2][2], int nqubits, int q)
{
	//n - число кубитов
        //q - номер кубита для преобразования

	int shift = nqubits-q;
	//Все биты нулевые, кроме соответствующего позиции преобразуемого кубита  
	int pow2q=1<<(shift);

	int N=1<<nqubits;
	for	(int i=0; i<N; i++)
	{
		//Обнуления меняющегося бита
		int i0 = i & ~pow2q;

		//Установка меняющегося бита
		int i1 = i | pow2q;

		//Получение значения меняющегося бита
		int iq = (i & pow2q) >> shift;

		out[i] = U[iq][0] * in[i0] + U[iq][1] *in[i1];
	}
}

```

Отметим, что существенная часть вычислений и логики кода приходится на битовые операции, однако, этого можно избежать: однокубитное преобразование в большинстве случаев является лишь подпрограммой и применяется к разным кубитам большое число раз. В 

In [25]:
for ex in examples:
    print(ex.title)

Участник:Danyanya/Алгоритм Ланцоша для точной арифметики (без переортогонализации)
Однокубитное преобразование вектора-состояния
Участник:Alexander34396/Обобщенный метод минимальных невязок
Участник:Maria Zaitseva/PAM (Partitioning Around Medoids)
Участник:VolkovNikita94/Алгоритм Ланцоша для точной арифметики (без переортогонализации)
Участник:Bulatral/QR-разложение плотной вещественной матрицы методом вращений Гивенса
Участник:Noite/EM-алгоритм кластеризации
Алгоритм Пурдома
Алговики:Используемые шаблоны
Алгоритм Холецкого
Участник:Igor.orpanen/Плотностный алгоритм кластеризации (DBSCAN)
Уравнение Пуассона, решение дискретным преобразованием Фурье
Участник:Антон Тодуа/Partitioning Around Medoids (PAM)
Участник:Kaholicl/Адаптивный крестовый метод
Алгоритм Беллмана-Форда
Участница:Sannikovats/Вычисление определенного интеграла с использованием адаптивно сгущающейся сетки (1)
Нахождение суммы элементов массива сдваиванием
Участник:Denemmy/Partitioning Around Medoids (Алгоритм)
Алгоритм К

# Autoprompt

In [26]:
import dspy

14:09:31 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
14:09:31 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


In [27]:

OPENAI_SERVER__KEY="ec5844f8ffa38262b7eed7fc3016e10294390a17952954f20237761a95b2fe74"
OPENAI_SERVER__URL="http://demo.labinform.ru:30101/v1"

In [28]:
from typing import List, Literal


class AnalyzerSerialComplexity(dspy.Signature):
    """
    Serial complexity of algorithm - the number of operations that need to be performed if the algorithm is executed serially. 
    
    Read the algorithm description and determine the Serial complexity.
    Provide exact latex formula in $O(n)$ notation with only $n$.
    """
    description: str = dspy.InputField()
    serial_complexity: str = dspy.OutputField()

class AnalyzerSerialInpSize(dspy.Signature):
    """
    Read the algorithm description and determine the Input data size.
    Provide exact latex formula in $O(n)$ notation with only $n$.
    """
    description: str = dspy.InputField()
    input_size: str = dspy.OutputField()

class AnalyzerSerialOutSize(dspy.Signature):
    """
    Read the algorithm description and determine the Output data size.
    Provide exact latex formula in $O(n)$ notation with only $n$.
    """
    description: str = dspy.InputField()
    output_size: str = dspy.OutputField()
    
class AnalyzerParallelFormHeight(dspy.Signature):
    """
    Parallel Form (PF) is a representation of an algorithm graph in which:
    - all vertices are divided into numbered subsets called layers;
    -the source vertex of every arc is located in a layer with a lower index than the destination vertex;
    - there are no arcs between vertices located within the same layer.
    
    The height of the PF is the total number of layers. 
    
    Read the algorithm description and determine the Parallel form height.
    Provide exact latex formula in $O(n)$ notation with only $n$.
    """
    description: str = dspy.InputField()
    form_height: str = dspy.OutputField()

    
class AnalyzerParallelFormWidth(dspy.Signature):
    """
    Parallel Form (PF) is a representation of an algorithm graph in which:
    - all vertices are divided into numbered subsets called layers;
    - the source vertex of every arc is located in a layer with a lower index than the destination vertex;
    - there are no arcs between vertices located within the same layer.
    
    The width of a layer is the number of vertices contained in that layer. The width of the LPF is the maximum width among all its layers.
    
    Read the algorithm description and determine the Parallel form width.
    Provide exact latex formula in $O(n)$ notation with only $n$.
    """
    description: str = dspy.InputField()
    form_width: str = dspy.OutputField()


class AlgoWikiAnalyzerMM(dspy.Module):
    def __init__(self):
        self.complexity_module = dspy.ChainOfThought(AnalyzerSerialComplexity)
        self.inputsize_module = dspy.ChainOfThought(AnalyzerSerialInpSize)
        self.outputsize_module = dspy.ChainOfThought(AnalyzerSerialOutSize)
        self.formheight_module = dspy.ChainOfThought(AnalyzerParallelFormHeight)
        self.formwidth_module = dspy.ChainOfThought(AnalyzerParallelFormWidth)
    
    def forward(self, message: str):
        complexity = self.complexity_module(description=message)
        inputsize = self.inputsize_module(description=message)
        outputsize = self.outputsize_module(description=message)
        formheight = self.formheight_module(description=message)
        formwidth = self.formwidth_module(description=message)

        reasonings = {
            "complexity":complexity.reasoning,
            "inp_size":inputsize.reasoning,
            "out_size":outputsize.reasoning,
            "form_height":formheight.reasoning,
            "form_width":formwidth.reasoning,
        }

        return dspy.Prediction(
            complexity=complexity.serial_complexity,
            inp_size=inputsize.input_size,
            out_size=outputsize.output_size,
            form_height=formheight.form_height,
            form_width=formwidth.form_width,
            reasoning = reasonings
        )

program = AlgoWikiAnalyzerMM()


In [30]:
gen_kwargs = {
    "temperature" : 1.0,
    "top_k" : 60,
    "top_p" : 0.8,
    "repetition_penalty": 1.0,
    "frequency_penalty":0.1,
    "min_p" : 0.2,
}

llm_name = "DeepSeek V3"

lm = dspy.LM(f"openai/{llm_name}", api_key=OPENAI_SERVER__KEY, api_base=OPENAI_SERVER__URL, model_type="chat", rollout_id=1)
dspy.configure(lm=lm)

In [31]:
lm.supported_params

{'audio',
 'extra_headers',
 'frequency_penalty',
 'function_call',
 'functions',
 'logit_bias',
 'logprobs',
 'max_completion_tokens',
 'max_retries',
 'max_tokens',
 'modalities',
 'n',
 'parallel_tool_calls',
 'prediction',
 'presence_penalty',
 'prompt_cache_key',
 'prompt_cache_retention',
 'response_format',
 'safety_identifier',
 'seed',
 'service_tier',
 'stop',
 'store',
 'stream',
 'stream_options',
 'temperature',
 'tool_choice',
 'tools',
 'top_logprobs',
 'top_p',
 'web_search_options'}

In [32]:
res = program(examples[0].code_desc)
res

Prediction(
    complexity='The serial complexity of the Lanczos algorithm is typically \\( O(n) \\) per iteration, where \\( n \\) is the number of non-zero elements in the matrix. For \\( k \\) iterations (where \\( k \\) is the number of desired eigenvalues), the total complexity is \\( O(kn) \\). Since \\( k \\) is often a small constant or proportional to \\( n \\), the complexity can be simplified to \\( O(n) \\) in many cases.',
    inp_size='$O(n^2)$ where $n$ is the dimension of the matrix.',
    out_size='\\( O(n) \\)',
    form_height='Not applicable (insufficient information to determine the Parallel Form height).',
    form_width='$O(n)$',
    reasoning={'complexity': 'The description provided discusses the implementation and scalability of the Lanczos algorithm, particularly focusing on parallel performance and optimization. However, it does not explicitly detail the algorithmic steps or mathematical operations involved, which are necessary to determine the serial complex

## O-notation evaluation

In [75]:
import re

vars_pattern = r"([^a-z\\\\]|^)([a-z])"
vars_pattern = re.compile(vars_pattern)

def fix_vars(text):
    return vars_pattern.sub(r"\1{\2} ", text)

In [76]:
import re
from pylatexenc.latexencode import unicode_to_latex

o_notation_pattern = "O\((.+)\)"
o_notation_pattern = re.compile(o_notation_pattern)

latex_fix_dict = {
    "\\text{log}" : "\\log",
    "log(" : "\\log(",
    "max(" : "\\max(",
    "min(" : "\\min("
}

def extract_o_notation(text, isolate_vars=True):
    text = text.replace("\\text{log}", "\\log").replace("log(", "\\log(")
    candidate = o_notation_pattern.findall(text)
    res = text.replace("$","").lower()
    if candidate:
        res = candidate[0].lower()
    if isolate_vars:
        res = fix_vars(res)
    return res

In [91]:
from sympy.parsing.latex import parse_latex
from sympy import lambdify
import numpy as np

from sklearn.metrics import r2_score

def logx(array:float, base:float):
    return np.log(array) / np.log(base)

def frac(nom, denom):
    return nom/denom

def np_lambdify(varname, func):
    # print(func.free_symbols)
    subdict = {vn : varname for vn in func.free_symbols}
    func = func.subs(subdict)
    # print(func)
    lamb = lambdify(varname, func, modules=[{'log': logx}, 'numpy'])
    if func.is_constant():
        return lambda t: np.full_like(t, lamb(t))
    else:
        return lambda t: lamb(np.array(t))

def o_notation_trajectory(formula, variable='n', eval_set = np.arange(1, 10+1)):
    expr = parse_latex(formula)
    try:
        calc = np_lambdify('n', expr)
        return calc(eval_set)
    except Exception as e:
        print(e,formula)
        return np.zeros_like(eval_set)
    

def o_notation_score(pr, gt):
    try:
        pr=extract_o_notation(pr)
        gt=extract_o_notation(gt)    
        pr = o_notation_trajectory(pr)
        gt = o_notation_trajectory(gt)
    except:
        print(pr, gt)
        score = 0

    
    # print(pr, gt)
    try:
        score = np.clip(r2_score(gt, pr),0,1)
    except:
        print(pr, gt)
        score = 0
    return score

In [92]:
def metric(example, pred, trace=None, pred_name=None, pred_trace=None):
    """
    Computes a score based on agreement between prediction and gold standard.
    Returns the score (float).
    """
    # Compute scores for all modules
    scores = [
        o_notation_score(
            getattr(pred,prop), getattr(example,prop)
        ) for prop in ['complexity', "inp_size", "out_size", "form_height", "form_width"]
    ]
        
    scores = scores
    # Overall score: average of the three accuracies
    total = np.mean(scores)
    pred['scores'] = scores

    return total


## Run bench

In [93]:
from dataclasses import asdict

def algowiki_to_dict(row, message_field='code_desc'):
    row = asdict(row)
    row['message'] = row.pop(message_field)
    row |= row.pop("table")
    return row

dataset = [
        dspy.Example(algowiki_to_dict(ex)).with_inputs("message")
        for ex in examples if ex.code_desc
]

In [94]:
for e, ex in enumerate(dataset):
    field = ex.out_size
    print(e, field.replace("$$", ""), "|||", extract_o_notation(field))
    print(o_notation_score(field,field))

0 k(n + 1) ||| {k} ({n}  + 1)
1.0
1 2^n ||| 2^{n} 
1.0
2 n^2 ||| {n} ^2
1.0
3 N^3 ||| {n} ^3
1.0
4 N ||| {n} 
1.0
5 n ||| {n} 
1.0
6 M*c ||| {m} *{c} 
1.0
7 n(n + 1) ||| {n} ({n}  + 1)
1.0
8 kn+k ||| {k} n+{k} 
1.0
9 3n-2 ||| 3{n} -2
1.0
10 nm ||| {n} m
1.0
11 n ||| {n} 
1.0
12 nk ||| {n} k
1.0
13 n^2 ||| {n} ^2
1.0
14 \frac{n (n + 1)}{2} ||| \frac{{n}  ({n}  + 1)}{2}
1.0
15 n ||| {n} 
1.0
16 2^n ||| 2^{n} 
1.0
17 m(n + 1) ||| {m} ({n}  + 1)
1.0
18 n ||| {n} 
1.0
19 n ||| {n} 
1.0
20 |V| ||| |{v} |
1.0
21 O(|V|) ||| |{v} |
1.0
22 n ||| {n} 
1.0
23 -1 ||| -1
1.0
24 ml ||| {m} l
1.0
25 n ||| {n} 
1.0
26 n ||| {n} 
1.0
27 mn ||| {m} n
1.0
28 n ||| {n} 
1.0
29 (n^2+3n)/2 ||| ({n} ^2+3{n} )/2
1.0
30 n ||| {n} 
1.0
31 n ||| {n} 
1.0
32 n ||| {n} 
1.0
33 1 ||| 1
1.0
34 O(k) ||| {k} 
1.0
35 n ||| {n} 
1.0
36 n ||| {n} 
1.0
37 n(n + 2) ||| {n} ({n}  + 2)
1.0
38 n(n + 1) ||| {n} ({n}  + 1)
1.0
39 2n ||| 2{n} 
1.0
40 (N_{x} + 1)(N_{y} + 1) ||| ({n} _{{x} } + 1)({n} _{{y} } + 1)
({n} _{{x} } + 1)(

In [95]:
import dspy

out_path = "./outputs/eval_results/20_05_26"
out_path = Path(out_path)
out_path.mkdir(exist_ok=True,parents=True)

to_eval = ["DeepSeek V3"] #["qwen3-4b-instruct", "qwen35_27b", "qwen35moe35b", "Qwen3-235B-A22B-Instruct-2507", "DeepSeek V3"]

for llm_name in tqdm(to_eval):
    lm = dspy.LM(f"openai/{llm_name}", api_key=OPENAI_SERVER__KEY, api_base=OPENAI_SERVER__URL, model_type="chat")
    dspy.configure(lm=lm)
    
    fp_name = llm_name.replace(" ","___")
    fp_full = out_path / f"{fp_name}.json"
    
    evaluate = dspy.Evaluate(
        devset=dataset,
        metric=metric,
        num_threads=10,
        display_table=True,
        display_progress=True,
        provide_traceback=True,
        save_as_json=fp_full
    )
    
    eval_res = evaluate(program)


  0%|                                                                                                                                                                     | 0/1 [00:00<?, ?it/s]

Average Metric: 2.58 / 8 (32.3%):  15%|██████████████████                                                                                                        | 8/54 [00:51<07:32,  9.83s/it][                   1    18014398509481984 -8127490854706933095
                    0  7378061867779487305 -6467169064904032256
 -2919578281501897711                    0  8331716707709345649
 -7908320945662590976] {k}  \\log({n} )
[                   1    18014398509481984 -8127490854706933095
                    0  7378061867779487305 -6467169064904032256
 -2919578281501897711                    0  8331716707709345649
 -7908320945662590976] {k}  \\log({n} )
Average Metric: 6.97 / 21 (33.2%):  37%|████████████████████████████████████████████▍                                                                           | 20/54 [01:23<01:42,  3.03s/it]logx() missing 1 required positional argument: 'base' {n} ^2+{n} m+{m} ^2\log({m} )
logx() missing 1 required positional argument: 'base' {n}  + \log({n

2026/06/11 14:29:19 INFO dspy.evaluate.evaluate: Average Metric: 18.371417776100266 / 54 (34.0%)


,title,math_desc,message,example_complexity,example_inp_size,example_out_size,example_form_height,example_form_width,pred_complexity,pred_inp_size,pred_out_size,pred_form_height,pred_form_width,reasoning,scores,metric
0,Участник:Danyanya/Алгоритм Ланцоша для точной арифметики (без пере...,# Общее описание алгоритма **Алгоритм Ланцоша поиска собственных з...,# Программная реализация алгоритма ## Масштабируемость алгоритма и...,$$O(kn^2)$$,$$\frac{n(n + 1)}{2}$$,$$k(n + 1)$$,$$O(k \log(n))$$,$$O(n^2)$$,"$O(mn)$ per iteration, where \( m \) is the number of non-zero ele...",The input size is given by the dimension of the square matrix: \n...,$O(n)$,The PF height cannot be determined from the provided description.,$O(n)$,{'complexity': 'The provided text describes a parallel implementat...,"[0.0, 0.0, 0.0, 0, 0.0]",✔️ [0.000]
1,Однокубитное преобразование вектора-состояния,# Общее описание алгоритма *Алгоритм производит моделирование дейс...,# Программная реализация алгоритма ## Особенности реализации после...,$$3 \cdot 2^n$$,$$2^n+4$$,$$2^n$$,$$2$$,$$2^{n+1}$$,$O(2^n)$,"The input size is \( O(2^n) \), where \( n \) is the number of qub...",\( O(2^n) \),$O(1)$,$O(1)$,"{'complexity': ""The algorithm performs a single-qubit transformati...","[0.36560986554023733, 0.9998366494182065, 1.0, 0.0, 0.0]",✔️ [0.473]
2,Участник:Bulatral/QR-разложение плотной вещественной матрицы метод...,# \n\n# \n\n# \n,# Программная реализация ## Масштабируемость алгоритма и его реали...,$$2n^3$$,$$n^2$$,$$n^2$$,$$11n-16$$,$$O(n^2)$$,$O(n^3)$,\( O(N^2) \),"The output size is \( O(n^2) \), as the algorithm performs operati...","$O\left(\frac{n}{k} + p\right)$, where $n$ is the matrix size, $k$...",$O(n)$,{'complexity': 'The algorithm described is a block-based QR decomp...,"[0.5348617684330308, 1.0, 1.0, 0.9385925369396444, 0.0]",✔️ [0.695]
3,"Уравнение Пуассона, решение дискретным преобразованием Фурье",# Общее описание алгоритма Уравнение Пуассона для многомерного про...,# Программная реализация алгоритма ## Особенности реализации после...,$$N^3(6\text{log}_2 N+1)$$,$$N^3$$,$$N^3$$,$$6N(\text{log}_2N)+1$$,$$N^2$$,The serial complexity of the algorithm is:\n\[ O(n^3 \log n) \],The input size is given by the total number of elements in the 3D ...,\(O(n)\),$O(1)$,The width of the Parallel Form is \(O(N^2)\).,{'complexity': 'The described algorithm involves performing a sequ...,"[0.0, 0.0, 0.0, 0.0, 1.0]",✔️ [0.200]
4,Участник:Denemmy/Partitioning Around Medoids (Алгоритм),# Общее описание алгоритма **Кластеризация** - это задача из облас...,# Программная реализация алгоритма ## Масштабируемость алгоритма и...,$$O(T*K*N^2)$$,$$N*(N-1)/2 + 2$$,$$N$$,-1,-1,$O(n^2)$,$O(n)$,$O(n^2)$,$O(1)$ (cannot be determined precisely from the given description),$O(n)$ (insufficient information to determine exact width),{'complexity': 'The algorithm described is the **PAM (Partitioning...,"[0.0, 0.0, 0.0, 0.0, 0.0]",✔️ [0.000]
5,Алгоритм Качмажа,# Общее описание алгоритма Метод алгебраической реконструкции - по...,# Программная реализация алгоритма Ниже представлен прототип прогр...,$$k \cdot O(n)$$,$$m \cdot n + m + n$$,$$n$$,-1,-1,"$O(m \cdot n)$, where $m$ is the number of rows in matrix $A$ and ...",$O(m \times n + m) = O(m \times n)$,$O(n)$,$O(n)$,$O(n)$,{'complexity': 'The provided algorithm is an implementation of the...,"[0.0, 0.9733941467122768, 1.0, 0.0, 0.0]",✔️ [0.395]
6,Участник:Gkhazeeva/Нечеткий алгоритм C средних,# Общее описание алгоритма Кластеризация - это объединение объекто...,# Программная реализация алгоритма ## Масштабируемость алгоритма и...,$$O(c^2 Mn_{iter} + cMdn_{iter})$$,$$M*d$$,$$M*c$$,$$O(cdn_{iter} + c^2n_{iter})$$,$$O (M)$$,$O(n)$,$O(n)$,$O(n)$,$O(1)$,$O(n)$,"{'complexity': ""The algorithm described is a parallel implementati...","[0.0, 0.0, 0.0, 0.0, 0.0]",✔️ [0.000]
7,Метод «разделяй и властвуй» вычисления собственных значений и вект...,# Общее описание алгоритма[[править](/w/ru/index.php?title=%D0%9C%...,# Программная реализация алгоритма[


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [10:21<00:00, 621.97s/it]


In [254]:
eval_res

[{'title': 'Однокубитное преобразование вектора-состояния',
  'math_desc': '# Общее описание алгоритма\n*Алгоритм производит моделирование действия однокубитного квантового вентиля на вектор-состояние.* [[1]](#cite_note-1) [[2]](#cite_note-2) [[3]](#cite_note-3) [[4]](#cite_note-4) Данный алгоритм обычно является подпрограммой и многократно применяется к различным кубитам одного состояния (например при моделировании квантовых алгоритмов или анализе квантовой запутанности). Особенностью алгоритма, как и большинства алгоритмов квантовой инфоорматики, является экспоненциальный рост объема данных в зависимости от основного параметра - числа кубитов, что приводит к необходимости суперкомпьютерной реализации для решения важных практических задач.\n\n\n# Математическое описание алгоритма\n**Исходные данные:**\n\nЦелочисленные параметры $n -$ число кубитов (необязательно) и $k -$ номер кубита, над которым производится преобразование.\n\nКомплексная матрица $U = \\begin{pmatrix}\nu_{00} & u_{01

In [228]:
import json


def load_eval_res(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

In [ ]:

str.re

In [270]:
import pandas as pd

size_dict = {
    "Qwen3-235B-A22B-Instruct-2507":235,
    "qwen35moe35b" :35,
    "qwen3-4b-instruct" : 4,
    "qwen35_27b" : 27,
    "DeepSeek___V3": 685
}

out_path = "./outputs/eval_results/20_05_26"
out_path = Path(out_path)
agg_rows = []
for fp_full in out_path.glob("*.json"):
    llm_name = fp_full.stem

    eval_res = load_eval_res(fp_full)
    rows = []
    for row in eval_res:
        columns = [k.removeprefix("example_") for k in row.keys() if k.startswith("example_")]
        rows.append(row['scores'])
    rows = np.mean(rows, axis=0)
    row = [llm_name] + [size_dict[llm_name]] + rows.tolist() + [rows.mean()]
    agg_rows.append(row)

df = pd.DataFrame(agg_rows, columns=["llm", "size"]+columns+['AVG']).round(3)
df.sort_values("size")

,llm,size,complexity,inp_size,out_size,form_height,form_width,AVG
2,qwen3-4b-instruct,4,0.294,0.301,0.624,0.259,0.533,0.402
3,qwen35_27b,27,0.294,0.434,0.749,0.267,0.533,0.456
1,qwen35moe35b,35,0.294,0.234,0.659,0.267,0.533,0.397
0,Qwen3-235B-A22B-Instruct-2507,235,0.294,0.301,0.725,0.267,0.400,0.397
4,DeepSeek___V3,685,0.294,0.392,0.659,0.200,0.267,0.362


In [306]:
out_path = "./outputs/eval_results/20_05_26"
out_path = Path(out_path)

fp_full = out_path / "qwen35_27b.json"
eval_res = load_eval_res(fp_full)

In [307]:
eval_res[0]

{'title': 'Однокубитное преобразование вектора-состояния',
 'math_desc': '# Общее описание алгоритма\n*Алгоритм производит моделирование действия однокубитного квантового вентиля на вектор-состояние.* [[1]](#cite_note-1) [[2]](#cite_note-2) [[3]](#cite_note-3) [[4]](#cite_note-4) Данный алгоритм обычно является подпрограммой и многократно применяется к различным кубитам одного состояния (например при моделировании квантовых алгоритмов или анализе квантовой запутанности). Особенностью алгоритма, как и большинства алгоритмов квантовой инфоорматики, является экспоненциальный рост объема данных в зависимости от основного параметра - числа кубитов, что приводит к необходимости суперкомпьютерной реализации для решения важных практических задач.\n\n\n# Математическое описание алгоритма\n**Исходные данные:**\n\nЦелочисленные параметры $n -$ число кубитов (необязательно) и $k -$ номер кубита, над которым производится преобразование.\n\nКомплексная матрица $U = \\begin{pmatrix}\nu_{00} & u_{01}\

In [315]:
from collections import defaultdict

cols = defaultdict(list)

for i in range(8,10):
    if i in [3]:
        continue
    ex = eval_res[i]
    for pref in ["example", "pred"]:
        cols['title'].append(ex['title'][:50])
        cols['name'].append(pref)
        for k in sorted(ex.keys()):
            if k.startswith(pref):
                cols[k.split("_", maxsplit=1)[1]].append("$${}$$".format(extract_o_notation(ex[k], isolate_vars=False)))
        cols['Score'].append(ex['metric'])

pd.DataFrame(cols).groupby(['title','name']).first()

complexity  \
title                                              name                 
Перемножение плотных неособенных матриц (последова example   $$2mnl$$   
                                                   pred       $$n^3$$   
Прогонка, точечный вариант                         example   $$8n-7$$   
                                                   pred         $$n$$   

                                                           form_height  \
title                                              name                  
Перемножение плотных неособенных матриц (последова example       $$n$$   
                                                   pred          $$n$$   
Прогонка, точечный вариант                         example    $$5n-4$$   
                                                   pred          $$n$$   

                                                           form_width  \
title                                              name                 
Перемножение плотных неособенных матриц (последова example    $$2ml$$   
                                                   pred       $$n^2$$   
Прогонка, точечный вариант                         example      $$2$$   
                                                   pred         $$1$$   

                                                             inp_size  \
title                                              name                 
Перемножение плотных неособенных матриц (последова example  $$mn+nl$$   
                                                   pred       $$n^2$$   
Прогонка, точечный вариант                         example   $$4n-2$$   
                                                   pred         $$n$$   

                                                           out_size  Score  
title                                              name                     
Перемножение плотных неособенных матриц (последова example   $$ml$$    0.2  
                                                   pred     $$n^2$$    0.2  
Прогонка, точечный вариант                         example    $$n$$    0.2  
                                                   pred       $$n$$    0.2

In [309]:
for k,v in eval_res[0]['reasoning'].items():
    text = [f"# {k.upper()}", v]
    display(Markdown("\n\n".join(text)))

# COMPLEXITY

The provided algorithm implements a single-qubit unitary transformation on a quantum state vector. The input size is defined by the number of qubits, denoted as $n$ (variable `nqubits` in the code). The size of the state vector, `N`, is calculated as $2^n$ (via the expression `1 << nqubits`). The core of the algorithm is a loop that iterates from 0 to `N`, performing a constant number of arithmetic and bitwise operations in each iteration to compute the output state vector components. Since the loop runs $2^n$ times and the work per iteration is $O(1)$, the total number of operations is proportional to the size of the state vector. Therefore, the serial complexity in terms of the number of qubits $n$ is exponential.

# INP_SIZE

The algorithm describes the implementation of a single-qubit quantum gate operation. The primary input data is the quantum state vector, represented by the array `in` in the provided C code. The size of this state vector depends on the number of qubits, denoted as $n$ (variable `nqubits` in the code). The code calculates the dimension of the vector as `N = 1 << nqubits`, which is equivalent to $2^n$. The text explicitly states that while auxiliary data (like indices) has linear size ($3n$), the processed data has exponential size. Therefore, the size of the input data (the state vector) scales exponentially with the number of qubits $n$.

# OUT_SIZE

The provided code implements a one-qubit evolution operation on a quantum state vector. The variable `nqubits` represents the number of qubits, denoted as $n$. The size of the state vector $N$ is calculated as `1 << nqubits`, which corresponds to $2^n$. The function iterates through all $N$ indices to compute the values for the output array `out`. Since the output represents the full quantum state vector, its size is determined by the number of amplitudes, which is $2^n$. Therefore, the output data size grows exponentially with the number of qubits. In Big-O notation with respect to $n$, this is expressed as $O(2^n)$.

# FORM_HEIGHT

The algorithm described implements a single-qubit gate operation on a quantum state vector of size $N = 2^n$. The core logic involves iterating through all $N$ elements of the state vector and updating each element `out[i]` based on two elements of the input vector `in[i0]` and `in[i1]`. 

In the context of the Parallel Form (PF) representation:
1.  **Vertices**: The vertices of the algorithm graph represent the computational operations. Specifically, there are input vertices (the state vector `in`), operation vertices (the calculation of each `out[i]`), and output vertices (the state vector `out`).
2.  **Dependencies**: The calculation of each `out[i]` depends only on the input values `in[i0]` and `in[i1]` and the constant matrix `U`. There are no dependencies between the calculation of `out[i]` and `out[j]` for $i \neq j$. The text explicitly mentions that the main loop can be parallelized ("распараллеливание основного цикла") and that indices can be precomputed to avoid sequential bit operations during the main computation.
3.  **Layers**: Since all output elements can be computed independently and simultaneously from the input elements, the dependency graph consists of input nodes at layer 0, operation nodes at layer 1, and output nodes at layer 2 (or simply operations at layer 1 if inputs/outputs are not counted as computation layers).
4.  **Height**: The height of the PF is the length of the longest path in the dependency graph. Since the operations are independent, the longest path is constant (Input $\to$ Operation $\to$ Output). It does not grow with the number of qubits $n$ or the vector size $N$.

Therefore, the parallel height is constant, independent of $n$. In Big-O notation, this is $O(1)$.

# FORM_WIDTH

The algorithm implements a single-qubit evolution on a quantum state vector. The size of the state vector is $N = 2^n$, where $n$ is the number of qubits. The main computational loop iterates over all $N$ components of the vector. Each iteration computes an output component `out[i]` based on input components `in[i0]` and `in[i1]`. There are no data dependencies between the iterations of the loop; each `out[i]` can be computed independently. Consequently, in the Parallel Form representation of the algorithm graph, all $N$ computational vertices can be placed in the same layer (or distributed such that the maximum layer width is $N$). Thus, the width of the Parallel Form is $2^n$. Expressed in Big-O notation with respect to $n$, the width is $O(2^n)$.

In [277]:
cols

defaultdict(list,
            {'name': ['example', 'pred'],
             'complexity': '$O(2^n)$',
             'form_height': '$O(1)$',
             'form_width': '$O(1)$',
             'inp_size': 'The input size is \\( O(2^n) \\), where \\( n \\) is the number of qubits.',
             'out_size': '\\( O(2^n) \\)'})

In [274]:
from IPython.display import display, Markdown, Latex
display(Markdown(eval_res[0]['example_complexity']))

$$3 \cdot 2^n$$

In [230]:
dspy.inspect_history()





[2026-05-21T03:57:24.325432]

System message:

Your input fields are:
1. `description` (str):
Your output fields are:
1. `reasoning` (str): 
2. `form_width` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## description ## ]]
{description}

[[ ## reasoning ## ]]
{reasoning}

[[ ## form_width ## ]]
{form_width}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Parallel Form (PF) is a representation of an algorithm graph in which:
        - all vertices are divided into numbered subsets called layers;
        - the source vertex of every arc is located in a layer with a lower index than the destination vertex;
        - there are no arcs between vertices located within the same layer.
        
        The width of a layer is the number of vertices contained in that layer. The width of the LPF is the maximum width among all its layers.
        
        Read the algorithm description and determi

In [47]:
print(examples[0].title)

Однокубитное преобразование вектора-состояния
